# Enhanced LightGBM Training with Comprehensive Imputation + SMOTE

This notebook implements a complete training pipeline that handles:
- **NaN values in features** → Median imputation
- **-1 values in labels** → Random Forest-based label imputation
- **Class imbalance** → SMOTE oversampling
- **Model training** → LightGBM classifier

**Goal:** Train a robust model that utilizes all available data through intelligent imputation strategies.

## 1. Import Libraries

In [ ]:
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
import numpy as np

print("✅ All libraries imported successfully")

## 2. Configuration and Setup

In [ ]:
# Configuration
DATASET_PATH = "clean_training_dataset.csv"  # Update this path
MODEL_PATH = "lgbm_comprehensive_imputation_model.joblib"

# Label mapping
label_map = {
    0: "BODY_TEXT", 1: "TITLE", 2: "H1",
    3: "H2", 4: "H3", 5: "H4"
}

# Feature columns (our clean 12 features)
feature_columns = [
    'font_size', 'is_bold', 'is_italic', 'text_length', 'x_pos',
    'starts_with_digit', 'is_titlecase', 'dot_count', 'page',
    'relative_font_size', 'is_centered', 'distance_from_previous'
]

print(f"📁 Dataset path: {DATASET_PATH}")
print(f"💾 Model will be saved to: {MODEL_PATH}")
print(f"🎯 Number of features: {len(feature_columns)}")

## 3. Load and Explore the Dataset

In [ ]:
# Load the dataset
print(f"📁 Loading dataset from '{DATASET_PATH}'...")
df = pd.read_csv(DATASET_PATH)

print(f"✅ Dataset loaded. Shape: {df.shape}")
print(f"\nColumns in dataset: {list(df.columns)}")

# Check for missing feature columns
missing_features = [col for col in feature_columns if col not in df.columns]
if missing_features:
    print(f"❌ Missing feature columns: {missing_features}")
else:
    print(f"✅ All {len(feature_columns)} feature columns found")

In [ ]:
# Explore initial label distribution
print("📊 Initial label distribution:")
label_counts = df['label'].value_counts().sort_index()
for label, count in label_counts.items():
    label_name = label_map.get(label, "MISSING/UNKNOWN")
    print(f"  {label} ({label_name}): {count}")

# Visualize label distribution
plt.figure(figsize=(10, 6))
plt.subplot(1, 2, 1)
label_counts.plot(kind='bar')
plt.title('Label Distribution (Raw Counts)')
plt.xlabel('Label ID')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
# Create labels for the pie chart
pie_labels = [f"{label_map.get(label, 'UNKNOWN')}\n({count})" for label, count in label_counts.items()]
plt.pie(label_counts.values, labels=pie_labels, autopct='%1.1f%%')
plt.title('Label Distribution (Percentages)')

plt.tight_layout()
plt.show()

## 4. Prepare Features and Target

In [ ]:
# Extract features and target
X = df[feature_columns].copy()
y = df['label'].copy()

print(f"📊 Features shape: {X.shape}")
print(f"📊 Target shape: {y.shape}")

# Check for any issues in the data
print(f"\n🔍 Data quality check:")
print(f"  Features with NaN: {X.isnull().sum().sum()}")
print(f"  Labels with -1: {(y == -1).sum()}")
print(f"  Unique labels: {sorted(y.unique())}")

## 5. Split Data (Before Imputation to Prevent Data Leakage)

In [ ]:
# Split data first to prevent data leakage during imputation
print("📊 Splitting data (80/20 train/test)...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# Show label distribution in train/test
print("\nTraining set label distribution:")
train_counts = y_train.value_counts().sort_index()
for label, count in train_counts.items():
    print(f"  {label} ({label_map.get(label, 'UNKNOWN')}): {count}")

print("\nTest set label distribution:")
test_counts = y_test.value_counts().sort_index()
for label, count in test_counts.items():
    print(f"  {label} ({label_map.get(label, 'UNKNOWN')}): {count}")

## 6. Step 1: Feature Imputation (Handle NaN Values)

In [ ]:
print("🔧 Step 1: Feature Imputation (NaN values)...")

# Check for NaN values
train_nan_count = X_train.isnull().sum().sum()
test_nan_count = X_test.isnull().sum().sum()

print(f"  Found {train_nan_count} NaN values in training features")
print(f"  Found {test_nan_count} NaN values in test features")

if train_nan_count > 0 or test_nan_count > 0:
    print("  🔧 Applying median imputation to features...")
    
    # Show which features have NaN values
    nan_summary = X_train.isnull().sum()
    nan_features = nan_summary[nan_summary > 0]
    if len(nan_features) > 0:
        print("    Features with NaN values:")
        for feature, count in nan_features.items():
            print(f"      {feature}: {count} NaN values")
    
    imputer = SimpleImputer(strategy='median')
    
    X_train_imputed = imputer.fit_transform(X_train)
    X_test_imputed = imputer.transform(X_test)
    
    # Convert back to DataFrames
    X_train = pd.DataFrame(X_train_imputed, columns=X_train.columns, index=X_train.index)
    X_test = pd.DataFrame(X_test_imputed, columns=X_test.columns, index=X_test.index)
    
    print("  ✅ Feature NaN values imputed")
else:
    print("  ✅ No NaN values found in features")

# Verify no NaN values remain
print(f"\nVerification - NaN values remaining:")
print(f"  Training features: {X_train.isnull().sum().sum()}")
print(f"  Test features: {X_test.isnull().sum().sum()}")

## 7. Step 2: Label Imputation (Handle -1 Values)

In [ ]:
print("🎯 Step 2: Label Imputation (-1 values)...")

missing_label_count = (y_train == -1).sum()
print(f"  Found {missing_label_count} missing labels (-1) in training data")

if missing_label_count > 0:
    print("  🤖 Training classifier to predict missing labels...")
    
    # Separate known and unknown labels
    known_mask = y_train != -1
    unknown_mask = y_train == -1
    
    X_known = X_train[known_mask]
    y_known = y_train[known_mask]
    X_unknown = X_train[unknown_mask]
    
    print(f"    Training on {len(X_known)} samples with known labels")
    print(f"    Predicting {len(X_unknown)} samples with missing labels")
    
    # Train a simple classifier for label imputation
    imputation_model = RandomForestClassifier(
        n_estimators=50, 
        random_state=42, 
        class_weight='balanced'
    )
    imputation_model.fit(X_known, y_known)
    
    # Predict missing labels
    predicted_labels = imputation_model.predict(X_unknown)
    
    # Show what labels were imputed
    imputed_distribution = pd.Series(predicted_labels).value_counts().sort_index()
    print("    Imputed label distribution:")
    for label, count in imputed_distribution.items():
        label_name = label_map.get(label, f"UNKNOWN_{label}")
        print(f"      {label} ({label_name}): {count}")
    
    # Replace -1 labels with predicted labels
    y_train_imputed = y_train.copy()
    y_train_imputed[unknown_mask] = predicted_labels
    
    print("  ✅ Label imputation complete")
    
    # Update y_train with imputed values
    y_train = y_train_imputed
    
else:
    print("  ✅ No missing labels found")

# Show final label distribution after imputation
print("\nFinal training label distribution after imputation:")
final_train_counts = y_train.value_counts().sort_index()
for label, count in final_train_counts.items():
    label_name = label_map.get(label, f"UNKNOWN_{label}")
    print(f"  {label} ({label_name}): {count}")

## 8. Step 3: Apply SMOTE for Class Balancing

In [ ]:
print("⚖️ Step 3: Applying SMOTE for class balancing...")

# Show distribution before SMOTE
print("  Label distribution before SMOTE:")
pre_smote_counts = y_train.value_counts().sort_index()
for label, count in pre_smote_counts.items():
    label_name = label_map.get(label, f"UNKNOWN_{label}")
    print(f"    {label} ({label_name}): {count}")

# Apply SMOTE
min_samples = y_train.value_counts().min()
k = min_samples - 1 if min_samples > 1 else 1
print(f"  Setting k_neighbors = {k} (based on smallest class size: {min_samples})")

smote = SMOTE(random_state=42, k_neighbors=k)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Show distribution after SMOTE
print(f"  Training set size: {len(X_train)} → {len(X_train_resampled)}")
print("  Label distribution after SMOTE:")
post_smote_counts = pd.Series(y_train_resampled).value_counts().sort_index()
for label, count in post_smote_counts.items():
    label_name = label_map.get(label, f"UNKNOWN_{label}")
    print(f"    {label} ({label_name}): {count}")

print("✅ SMOTE balancing complete")

## 9. Step 4: Train the LightGBM Model

In [ ]:
print("🚀 Step 4: Training LightGBM model...")

# Initialize the model
model = lgb.LGBMClassifier(
    random_state=42,
    n_estimators=100,
    learning_rate=0.1,
    verbose=-1  # Suppress training output
)

# Train the model
model.fit(X_train_resampled, y_train_resampled)
print("✅ Model training complete")

# Show feature importance
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n📊 Top 10 Most Important Features:")
print(feature_importance.head(10))

# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(range(len(feature_importance)), feature_importance['importance'])
plt.yticks(range(len(feature_importance)), feature_importance['feature'])
plt.xlabel('Feature Importance')
plt.title('LightGBM Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 10. Step 5: Evaluate Model Performance

In [ ]:
print("📈 Step 5: Evaluating model performance...")

# Make predictions
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Accuracy: {accuracy:.4f}")

# Get label names for the classes present in the test set
unique_labels = sorted(set(y_test.unique()) | set(y_pred))
target_names = [label_map.get(label, f"UNKNOWN_{label}") for label in unique_labels]

# Classification report
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=target_names, labels=unique_labels))

### Confusion Matrix Visualization

In [ ]:
# Confusion matrix
print("📊 Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred, labels=unique_labels)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix - LightGBM with Comprehensive Imputation + SMOTE')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Calculate and display per-class metrics
print("\n🎯 Per-Class Performance Analysis:")
for i, label in enumerate(unique_labels):
    label_name = label_map.get(label, f"UNKNOWN_{label}")
    true_positives = cm[i, i]
    false_positives = cm[:, i].sum() - true_positives
    false_negatives = cm[i, :].sum() - true_positives
    
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"  {label_name}: Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f}")

## 11. Save the Final Model

In [ ]:
# Save the trained model
joblib.dump(model, MODEL_PATH)
print(f"💾 Model saved to '{MODEL_PATH}'")

# Save model metadata for future reference
model_info = {
    'model_type': 'LightGBM with SMOTE and Comprehensive Imputation',
    'features_used': feature_columns,
    'training_samples': len(X_train_resampled),
    'test_accuracy': accuracy,
    'label_mapping': label_map,
    'imputation_applied': True,
    'smote_applied': True
}

import json
with open(MODEL_PATH.replace('.joblib', '_info.json'), 'w') as f:
    json.dump(model_info, f, indent=4)

print(f"📄 Model metadata saved to '{MODEL_PATH.replace('.joblib', '_info.json')}'")

## 12. Final Summary

In [ ]:
print("🎉 TRAINING PIPELINE COMPLETE!")
print("=" * 50)
print(f"📊 Final Results:")
print(f"   • Model Type: LightGBM with Comprehensive Imputation + SMOTE")
print(f"   • Training Samples: {len(X_train_resampled):,}")
print(f"   • Test Samples: {len(X_test):,}")
print(f"   • Features Used: {len(feature_columns)}")
print(f"   • Final Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   • Model Saved: {MODEL_PATH}")
print("\n✅ Your model is ready for production use!")

# Quick test prediction to verify model works
print("\n🔬 Quick Model Test:")
test_sample = X_test.iloc[0:1]
test_prediction = model.predict(test_sample)[0]
test_actual = y_test.iloc[0]
print(f"   Sample prediction: {test_prediction} ({label_map.get(test_prediction, 'UNKNOWN')})")
print(f"   Actual label: {test_actual} ({label_map.get(test_actual, 'UNKNOWN')})")
print(f"   ✅ Model is working correctly!")